# DATASCI 530 – Project 1: MPIDR Research Projects

## Project description

**Source.** The research pages of the Max Planck Institute for Demographic
Research (MPIDR), https://www.demogr.mpg.de/en/research_6120/. 

**Datasets.**
- `mpidr_projects.csv` has **one row per research project** (129 rows). It
  records where the project sits in the institute (unit group, unit, research
  area), its title, its MPIDR team members, how many external collaborators and
  institutions it works with, the length of its short description, its research
  and region keywords, and the publications listed on the project page (count,
  open-access count, earliest and latest publication year).
- `mpidr_project_members.csv` has **one row per project × MPIDR member**
  (383 rows, 99 different researchers). A researcher on several projects
  appears once per project. It covers MPIDR staff only, because only they link
  to the staff directory; external collaborators are counted in the projects
  file but not named.
- `mpidr_publications.csv` has **one row per unit × publication** (1,967 rows,
  1,710 different publications). It comes from each unit's *Publications* tab,
  which lists far more than the project pages: many papers are not linked to
  any project. It records the unit, title, authors, publication type, year,
  open access, and the publication's URL.

**Scope.**
- Collected on **26 October 2026** (column `collected_on`) in one run of about
  150 page loads and 8.5 minutes.
- 17 research units across 4 groups: the Social Demography lab, the Digital and
  Computational Demography lab, the joint research labs and the independent
  research groups. 14 units list projects, spread over 20 research areas.
- 1,062 publications are listed on the project pages, published between
  **2004 and 2026**. Among projects with publications, half have their first one in
  2022 or later, so the data mostly describe recent and ongoing work.
- The unit *Publications* tabs were collected on **28 October 2026**. All 17
  units have one, including the three units without projects.
- The site shows no project start or end dates, so publication years are the
  only time information.

## Setup

In [1]:
# Executes the python script
exec(open("./scripts/import_packages.py").read())

BASE  = "https://www.demogr.mpg.de"
DELAY = 1.5    # seconds to wait after each page load

# Folder for diagnostic HTML files
os.makedirs("html_files", exist_ok = True)

Initialize the web driver

In [2]:
options = webdriver.ChromeOptions()

# False shows the browser window; True hides it
headless = False
if headless:
    options.add_argument("--headless=new")

driver = webdriver.Chrome(options = options)

Helper functions

- `open_page()` opens a URL and then waits `DELAY` seconds
- `save_diagnostic()` prettifies an element's HTML with BeautifulSoup and
  writes it to `html_files/`.
- `get_text()` reads the text of an element. Selenium's `.text` returns an
  empty string for elements that are hidden on screen, so we read the `textContent` attribute instead

In [3]:
def open_page(url):
    driver.get(url)
    time.sleep(DELAY)

def save_diagnostic(element, filename):
    html_code  = element.get_attribute("outerHTML")
    parse_code = BeautifulSoup(html_code, "html.parser").prettify()
    with open("html_files/" + filename, "w") as file:
        file.write(parse_code)

def get_text(element):
    # textContent also works for hidden elements; split() tidies whitespace
    return " ".join(element.get_attribute("textContent").split())

## Step 1: Find all research units

In Chrome's Developer Tools, the labs and research groups are listed in the
*Research* drop-down menu. Each unit is a link inside
`<li class="nav-level-3s">`, so the XPath is

`//li[@class = "nav-level-3s"]/a`

In [4]:
open_page(BASE + "/en/research_6120/")

list_unit_links = driver.find_elements(
    'xpath', '//li[@class = "nav-level-3s"]/a')
print(len(list_unit_links), "unit links found")

# Save the first link for diagnostics
save_diagnostic(list_unit_links[0], "diagnose_unit_link.html")

17 unit links found


In [5]:
units = []
seen  = []
for i in range(len(list_unit_links)):
    # get_attribute("href") returns the full URL
    unit_url = list_unit_links[i].get_attribute("href").rstrip("/") + "/"

    # The same menu can appear twice on a page, so skip duplicates
    if unit_url in seen:
        continue
    seen.append(unit_url)

    # The URL looks like /en/research_6120/<unit_group>/<unit>/
    unit_group = unit_url.split("/")[-3]

    units.append({"unit_url":   unit_url,
                  "unit_group": unit_group,
                  "unit":       get_text(list_unit_links[i])})

units = pd.DataFrame(units)
units

,unit_url,unit_group,unit
0,https://www.demogr.mpg.de/en/research_6120/soc...,social_demography_myrskylae_11664,Fertility and Well-Being
1,https://www.demogr.mpg.de/en/research_6120/soc...,social_demography_myrskylae_11664,Population Health
2,https://www.demogr.mpg.de/en/research_6120/soc...,social_demography_myrskylae_11664,Research Group: Mortality Inequalities
3,https://www.demogr.mpg.de/en/research_6120/soc...,social_demography_myrskylae_11664,Research Group: Labor Demography
4,https://www.demogr.mpg.de/en/research_6120/soc...,social_demography_myrskylae_11664,BIOSFER
5,https://www.demogr.mpg.de/en/research_6120/soc...,social_demography_myrskylae_11664,Max Planck – University of Helsinki Center for...
6,https://www.demogr.mpg.de/en/research_6120/soc...,social_demography_myrskylae_11664,Partner Group Uruguay: Simulation of Socio-Dem...
7,https://www.demogr.mpg.de/en/research_6120/soc...,social_demography_myrskylae_11664,Partner Group Malawi: Family Demography in Sub...
8,https://www.demogr.mpg.de/en/research_6120/dig...,digital_and_computational_demography_zagheni_1...,Migration and Mobility
9,https://www.demogr.mpg.de/en/research_6120/dig...,digital_and_computational_demography_zagheni_1...,Population Dynamics and Sustainable Well-Being


## Step 2: Loop over each unit's project list

Each unit has a *Projects* tab at `<unit_url>projects/`. In Developer Tools:

- every project is a `<div class="projectslist-item">` that holds the project
  title and a "Project details" link
- in labs, projects are grouped under research area headings: the area is the
  closest `<h2>` above the project's list, found with the XPath
  `../preceding-sibling::h2[1]`. Research groups have no research areas, so
  this is missing for them

Units with no projects on that tab are recorded in `skipped`.

In [6]:
# Test on a single unit first
open_page(units.loc[0, "unit_url"] + "projects/")

list_items = driver.find_elements(
    'xpath', '//div[@class = "projectslist-item"]')
print(len(list_items), "projects in", units.loc[0, "unit"])

save_diagnostic(list_items[0], "diagnose_project_item.html")

10 projects in Fertility and Well-Being


In [7]:
project_links = []
skipped       = []

for i in range(len(units)):
    open_page(units.loc[i, "unit_url"] + "projects/")

    list_items = driver.find_elements(
        'xpath', '//div[@class = "projectslist-item"]')

    if len(list_items) == 0:
        skipped.append(units.loc[i, "unit"])
        continue

    for item in list_items:
        link = item.find_element('xpath', './a')
        area = item.find_elements('xpath', '../preceding-sibling::h2[1]')

        project_links.append({
            "project_url":   link.get_attribute("href"),
            "unit":          units.loc[i, "unit"],
            "unit_group":    units.loc[i, "unit_group"],
            "research_area": get_text(area[0]) if len(area) > 0 else None})

projects = pd.DataFrame(project_links)
print(len(projects), "project links;", "skipped units:", skipped)

129 project links; skipped units: ['Partner Group Uruguay: Simulation of Socio-Demographic Systems', 'Partner Group Malawi: Family Demography in Sub-Saharan Africa', 'Reproductive Ageing']


## Step 3: Navigate to each project page

The project content is inside `<div class="box-page-content">`. From Developer
Tools, the parts we need are:

| Variable | XPath (inside the content box) |
|---|---|
| Title | `h2[1]` (first `<h2>`) |
| Team paragraph | `h2[1]/following-sibling::p[1]` |
| MPIDR members | links in the team paragraph to `staff_directory` |
| Description | `h2[1]/following-sibling::p[2]` |
| Keywords / regions | `<p class="lab-project-keywords-value">` next to its label |
| Publications | `<div class="publications-item-smaller">` |
| Open access | `<img title="Open Access">` inside a publication |
| Publication year | last `(YYYY)` in a publication's `<div class="bibliography">` |

The team paragraph lists MPIDR people first, then "in Collaboration with"
and external collaborators as `Name (Institution, Country)`.

In [8]:
CONTENT = '//div[@class = "box-page-content"]'

def keyword_value(label):
    """Value next to a label such as 'Region keywords:'."""
    found = driver.find_elements(
        'xpath',
        CONTENT + '//p[@class = "lab-project-keywords-label"]'
        + '[normalize-space() = "' + label + '"]'
        + '/../following-sibling::div[1]/p')
    return get_text(found[0]) if len(found) > 0 else None

def parse_project(url):
    open_page(url)

    title = driver.find_elements('xpath', CONTENT + '/h2[1]')
    team  = driver.find_elements(
        'xpath', CONTENT + '/h2[1]/following-sibling::p[1]')
    desc  = driver.find_elements(
        'xpath', CONTENT + '/h2[1]/following-sibling::p[2]')

    # MPIDR members: names that link to the staff directory
    members = []
    if len(team) > 0:
        members = team[0].find_elements(
            'xpath', './/a[contains(@href, "staff_directory")]')

    # External collaborators: text after "in Collaboration with"
    team_text = get_text(team[0]) if len(team) > 0 else ""
    collab = re.split(r"[Cc]ollaboration with", team_text)
    collab = collab[1] if len(collab) > 1 else ""
    institutions = re.findall(r"\(([^)]*)\)", collab)
    ext_names = [n for n in re.sub(r"\([^)]*\)", "", collab).split(",")
                 if n.strip()]

    # Description length, without the "Detailed Description" link text
    desc_words = None
    if len(desc) > 0:
        desc_text  = get_text(desc[0]).replace("Detailed Description", "")
        desc_words = len(desc_text.split())

    # Publications, open access, and earliest / latest year
    pubs = driver.find_elements(
        'xpath', CONTENT + '//div[@class = "publications-item-smaller"]')
    n_open = len(driver.find_elements(
        'xpath', CONTENT + '//div[@class = "publications-item-smaller"]'
        + '//img[@title = "Open Access"]'))
    years = []
    for pub in pubs:
        bib = pub.find_elements('xpath', './div[@class = "bibliography"]')
        if len(bib) > 0:
            # The publication year is the last "(YYYY)" in the citation
            found = re.findall(r"\((\d{4})\)", get_text(bib[0]))
            if len(found) > 0:
                years.append(int(found[-1]))

    return {"project_url":  url,
            "title":        get_text(title[0]) if len(title) > 0 else None,
            "members":      "; ".join(get_text(a) for a in members),
            "member_urls":  "; ".join(a.get_attribute("href")
                                      for a in members),
            "n_members":    len(members),
            "n_external":   len(ext_names),
            "n_external_institutions": len(institutions),
            "description_words": desc_words,
            "keywords":     keyword_value("Research Keywords:"),
            "regions":      keyword_value("Region keywords:"),
            "n_publications": len(pubs),
            "n_open_access":  n_open,
            "earliest_pub_year": min(years) if len(years) > 0 else None,
            "latest_pub_year":   max(years) if len(years) > 0 else None}

In [9]:
# Test on a single project first
test = parse_project(projects.loc[0, "project_url"])
save_diagnostic(driver.find_element('xpath', CONTENT),
                "diagnose_project_page.html")
test

{'project_url': 'https://www.demogr.mpg.de/en/research_6120/social_demography_myrskylae_11664/fertility_and_well_being_3923/contemporary_and_future_trends_in_fertility_4843/shifts_in_the_fertility_development_nexus_at_the_macro_and_micro_level_4992',
 'title': 'Shifts in the Fertility–Development Nexus at the Macro and Micro Level',
 'members': 'Aiva Jasilioniene; Daniel Ciganda; Andres Castro; Mikko Myrskylä',
 'member_urls': 'https://www.demogr.mpg.de/en/about_us_6113/staff_directory_1899/aiva_jasilioniene_1265; https://www.demogr.mpg.de/en/about_us_6113/staff_directory_1899/daniel_ciganda_2829; https://www.demogr.mpg.de/en/about_us_6113/staff_directory_1899/andres_castro_3852; https://www.demogr.mpg.de/en/about_us_6113/staff_directory_1899/mikko_myrskylae_1747',
 'n_members': 4,
 'n_external': 3,
 'n_external_institutions': 3,
 'description_words': 34,
 'keywords': 'Fertility Development',
 'regions': 'OECD countries, World',
 'n_publications': 29,
 'n_open_access': 28,
 'earliest_p

In [10]:
# Loop over every unique project (a project can be listed by two units)
unique_urls = projects["project_url"].unique()

details = []
for i in range(len(unique_urls)):
    details.append(parse_project(unique_urls[i]))

details = pd.DataFrame(details)

## Step 4: Collect each unit's publication list

The project pages only list publications linked to a project. Each unit also
has a *Publications* tab with its full list. For every unit, the scraper opens
the unit page and **clicks** the *Publications* button
(`//a[@class = "buttonfilter"][normalize-space() = "Publications"]`).

Each entry is again a `<div class="publications-item-smaller">`:

| Variable | Source inside the entry |
|---|---|
| Title, URL | `div[@class = "title"]/a` |
| Authors | `div[@class = "authors"]` |
| Type | folder in the URL, e.g. `.../journal_articles/...`; entries that link to an outside DOI are labelled "external link" |
| Year | last `(YYYY)` in `div[@class = "bibliography"]` |
| Open access | `<img title="Open Access">` |

In [ ]:
def parse_publication(pub):
    """Title, authors, type, year and open access of one publication entry."""
    title = pub.find_elements('xpath', './div[@class = "title"]/a')
    auth  = pub.find_elements('xpath', './div[@class = "authors"]')
    bib   = pub.find_elements('xpath', './div[@class = "bibliography"]')

    url   = title[0].get_attribute("href") if len(title) > 0 else None
    years = re.findall(r"\((\d{4})\)", get_text(bib[0])) if len(bib) > 0 else []

    # MPIDR URLs look like .../publications_1904/<type>/<title>_<id>/
    if url and "publications_1904" in url:
        pub_type = url.rstrip("/").split("/")[-2]
    else:
        pub_type = "external link"

    return {"pub_title":   get_text(title[0]) if len(title) > 0 else None,
            "authors":     get_text(auth[0]).rstrip(":") if len(auth) > 0 else None,
            "pub_type":    pub_type,
            "pub_year":    int(years[-1]) if len(years) > 0 else None,
            "open_access": len(pub.find_elements(
                               'xpath', './/img[@title = "Open Access"]')) > 0,
            "pub_url":     url}

PUB_TAB  = '//a[@class = "buttonfilter"][normalize-space() = "Publications"]'
PUB_ITEM = CONTENT + '//div[@class = "publications-item-smaller"]'

In [ ]:
# Test on a single unit first
open_page(units.loc[0, "unit_url"])
driver.find_element('xpath', PUB_TAB).click()
time.sleep(DELAY)

list_pubs = driver.find_elements('xpath', PUB_ITEM)
print(len(list_pubs), "publications in", units.loc[0, "unit"])

save_diagnostic(list_pubs[0], "diagnose_unit_publication.html")
parse_publication(list_pubs[0])

In [ ]:
unit_pubs = []
no_tab    = []

for i in range(len(units)):
    open_page(units.loc[i, "unit_url"])

    # Click the unit's Publications button
    tab = driver.find_elements('xpath', PUB_TAB)
    if len(tab) == 0:
        no_tab.append(units.loc[i, "unit"])
        continue
    tab[0].click()
    time.sleep(DELAY)

    for pub in driver.find_elements('xpath', PUB_ITEM):
        row = parse_publication(pub)
        row["unit"]       = units.loc[i, "unit"]
        row["unit_group"] = units.loc[i, "unit_group"]
        unit_pubs.append(row)

unit_pubs = pd.DataFrame(unit_pubs)
print(len(unit_pubs), "publication rows;", "units without a tab:", no_tab)

# Close the browser when finished
driver.quit()

## Step 5: Build and save the datasets

| File | One row per | Main variables |
|---|---|---|
| `mpidr_projects.csv` | project | unit, research area, title, team, keywords, regions, counts below |
| `mpidr_project_members.csv` | project × MPIDR member | title, unit, research area, member, member profile URL |
| `mpidr_publications.csv` | unit × publication | unit, title, authors, type, year, open access, URL |

Count variables: `n_members` (MPIDR team members), `n_external` (external
collaborators), `n_external_institutions`, `description_words`,
`n_publications`, `n_open_access`, `earliest_pub_year`, and `latest_pub_year`.

A publication can appear in several units, so `pub_url` identifies distinct
publications.

In [11]:
out = projects.merge(details, on = "project_url", how = "left")
out["collected_on"] = date.today().isoformat()
out.to_csv("mpidr_projects.csv", index = False)

# One row per project x MPIDR member (useful for finding researchers by topic)
has_members = out[out["n_members"] > 0]
members = (has_members
           .assign(member     = has_members["members"].str.split("; "),
                   member_url = has_members["member_urls"].str.split("; "))
           .explode(["member", "member_url"])
           [["title", "unit", "research_area", "member", "member_url"]])
members.to_csv("mpidr_project_members.csv", index = False)

# One row per unit x publication, from the unit Publications tabs
unit_pubs["collected_on"] = date.today().isoformat()
unit_pubs = unit_pubs[["unit_group", "unit", "pub_title", "authors", "pub_type",
                       "pub_year", "open_access", "pub_url", "collected_on"]]
unit_pubs.to_csv("mpidr_publications.csv", index = False)

print(len(out), "project rows;",
      out["project_url"].nunique(), "unique projects;",
      len(members), "member rows;",
      len(unit_pubs), "publication rows")

129 project rows; 129 unique projects; 383 member rows


## Step 6: Quality checks

In [12]:
data    = pd.read_csv("mpidr_projects.csv")
members = pd.read_csv("mpidr_project_members.csv")
collect_year = int(data["collected_on"].iloc[0][:4])

checks = {
    "Every project URL is unique":
        data["project_url"].is_unique,
    "Every project has a title":
        data["title"].notna().all(),
    "Open-access count never exceeds publications":
        (data["n_open_access"] <= data["n_publications"]).all(),
    "Years missing only when there are no publications":
        (data["latest_pub_year"].isna() == (data["n_publications"] == 0)).all(),
    "Earliest year is never after the latest year":
        (data["earliest_pub_year"].dropna() <= data["latest_pub_year"].dropna()).all(),
    "Publication years are not after the collection year":
        (data["latest_pub_year"].dropna() <= collect_year).all(),
    "External institutions never exceed external collaborators":
        (data["n_external_institutions"] <= data["n_external"]).all(),
    "Member file has one row per listed member":
        len(members) == data["n_members"].sum()}

pd.Series(checks, name = "passed").to_frame()

,passed
Every project URL is unique,True
Every project has a title,True
Open-access count never exceeds publications,True
Years missing only when there are no publications,True
Earliest year is never after the latest year,True
Publication years are not after the collection year,True
External institutions never exceed external collaborators,True
Member file has one row per listed member,True


In [ ]:
pubs = pd.read_csv("mpidr_publications.csv")
pub_year = int(pubs["collected_on"].iloc[0][:4])

pub_checks = {
    "Every publication has a title, URL and year":
        pubs[["pub_title", "pub_url", "pub_year"]].notna().all().all(),
    "No publication is listed twice within a unit":
        not pubs.duplicated(["unit", "pub_url"]).any(),
    "All 17 units have a publication list":
        pubs["unit"].nunique() == 17,
    # Forthcoming articles can carry next year's date
    "Publication years are at most one year after collection":
        (pubs["pub_year"] <= pub_year + 1).all()}

pd.Series(pub_checks, name = "passed").to_frame()

In [13]:
# Projects without any MPIDR member links
data.loc[data["n_members"] == 0, ["unit", "title"]]

,unit,title
79,Population Dynamics and Sustainable Well-Being,Estimating the Health Effects of Violence and ...


All checks pass. One
project lists no team on its page, so its member fields are empty; it is kept
because its other information is complete.

In the publications file, 3 forthcoming journal articles are dated 2027. 19
BIOSFER entries link to an outside DOI instead of an MPIDR page, so their type
is "external link". Some papers are listed twice under different URLs, once as
an MPIDR working paper and once as the journal article: there are 1,710
distinct URLs but only 1,617 distinct titles.

## Step 7: Scope, observations, missing values, mean, and variance

In [14]:
scope = pd.Series({
    "Collection date":              data["collected_on"].iloc[0],
    "Research units with projects": f'{data["unit"].nunique()} of 17',
    "Research areas":               data["research_area"].nunique(),
    "Projects":                     len(data),
    "MPIDR researchers":            members["member"].nunique(),
    "Publications on project pages": data["n_publications"].sum(),
    "Publications on unit tabs (distinct)": pubs["pub_url"].nunique(),
    "Units with a publication list": pubs["unit"].nunique(),
    "Publication years":            f'{data["earliest_pub_year"].min():.0f}'
                                    f'–{data["latest_pub_year"].max():.0f}'})
scope.to_frame("value")

,value
Collection date,2026-10-26
Research units with projects,14 of 17
Research areas,20
Projects,129
MPIDR researchers,99
Publications listed,1062
Publication years,2004–2026


In [15]:
numeric = ["n_members", "n_external", "n_external_institutions",
           "description_words", "n_publications", "n_open_access",
           "earliest_pub_year", "latest_pub_year"]

summary = pd.DataFrame({"observations": data.notna().sum(),
                        "missing":      data.isna().sum()})
summary["mean"]     = data[numeric].mean().round(2)
summary["variance"] = data[numeric].var().round(2)

print(len(data), "projects;", len(members), "project-member rows;",
      members["member"].nunique(), "unique MPIDR researchers")
summary.fillna("")

129 projects; 383 project-member rows; 99 unique MPIDR researchers


,observations,missing,mean,variance
project_url,129,0,,
unit,129,0,,
unit_group,129,0,,
research_area,102,27,,
title,129,0,,
members,128,1,,
member_urls,128,1,,
n_members,129,0,2.97,2.45
n_external,129,0,1.72,6.42
n_external_institutions,129,0,1.47,4.45


## Step 8: How severe are the missing values, and what are they related to?

The table above shows missing values in `keywords`, `regions` and the two
publication-year columns (the earliest and latest years are always missing
together). Below I measure how common each problem is, whether the problems
occur together, and whether they are linked to the unit or to other variables.

In [16]:
check   = ["keywords", "regions", "latest_pub_year"]
missing = data[check].isna()

severity = pd.DataFrame({"projects missing": missing.sum(),
                         "% of projects":    (100 * missing.mean()).round(1)})
severity

,projects missing,% of projects
keywords,8,6.2
regions,31,24.0
latest_pub_year,14,10.9


In [17]:
# Do keywords and regions go missing together?
pd.crosstab(missing["keywords"], missing["regions"], margins = True).rename_axis(
    index = "keywords missing", columns = "regions missing")

regions missing,False,True,All
keywords missing,,,
False,97,24,121
True,1,7,8
All,98,31,129


In [18]:
# Share of projects with each problem (%), by unit group
group_names = {
    "social_demography_myrskylae_11664":                  "Social Demography lab",
    "digital_and_computational_demography_zagheni_11666": "Digital & Computational lab",
    "joint_research_labs_11667":                          "Joint research labs",
    "independent_research_groups_11668":                  "Independent research groups"}
unit_type = data["unit_group"].map(group_names)

by_type = missing.groupby(unit_type).mean().mul(100).round(1)
by_type["projects"] = unit_type.value_counts()
by_type.rename_axis("unit group")

,keywords,regions,latest_pub_year,projects
unit group,,,,
Digital & Computational lab,1.8,3.6,14.3,56
Independent research groups,5.9,23.5,11.8,17
Joint research labs,0.0,15.4,0.0,13
Social Demography lab,14.0,53.5,9.3,43


In [19]:
# Average project characteristics when each field is present vs. missing
data["dissertation_share"] = data["title"].str.contains("(Dissertation)", regex = False)
traits = ["n_members", "n_external", "description_words",
          "n_publications", "earliest_pub_year", "dissertation_share"]

compare = pd.concat(
    {v: data.groupby(missing[v].map({False: "present", True: "missing"}))[traits].mean()
     for v in check})
compare.rename_axis(["field", "status"]).round(2)

n_members  n_external  description_words  \
field           status                                              
keywords        missing       3.38        0.38              32.25   
                present       2.94        1.81              39.07   
regions         missing       2.65        1.03              38.32   
                present       3.07        1.94              38.74   
latest_pub_year missing       2.36        0.36              35.14   
                present       3.04        1.89              39.07   

                         n_publications  earliest_pub_year  dissertation_share  
field           status                                                          
keywords        missing            7.62            2021.67                0.50  
                present            8.27            2020.70                0.21  
regions         missing           11.10            2021.03                0.23  
                present            7.33            2020.65                0.22  
latest_pub_year missing            0.00                NaN                0.79  
                present            9.23            2020.75                0.16

**Severity.**
- `regions` is the most serious problem: 31 projects (24%) have no region tag.
  It is not just non-geographic work. Several untagged projects name a place in
  their title (for example *Health Outcomes of Automation Risk in Germany* and
  *Union–Fertility Nexus ... in Sub-Saharan Africa*), so any regional analysis
  in Project 2 will undercount these regions.
- `keywords` is a minor problem: 8 projects (6%). In 7 of the 8 the whole
  keyword box is absent, so regions are missing too.
- `latest_pub_year` / `earliest_pub_year` (14 projects, 11%) are not errors.
  These projects list no publications, so a year cannot exist. They should be
  left out of year-based summaries, not filled in.

**What the missing values are related to.**
- **The unit, most of all.** More than half of the Social Demography lab's
  projects (54%) lack regions and 14% lack keywords, against 4% and 2% in the
  Digital & Computational lab. This points to different editing habits across
  units, not to scraping errors. Within the Social Demography lab, Population
  Health (8 of 14) and Labor Demography (6 of 6) account for most of the gap.
- **Project size and age matter little for keywords and regions.** Projects
  with and without these tags have similar team sizes, description lengths
  and first publication years. Projects without keywords have fewer external
  collaborators (0.4 vs. 1.8) and are more often dissertations (50% vs. 21%),
  but this rests on only 8 projects.
- **Projects without publications are mostly dissertations.** 11 of the 14
  are dissertation projects (vs. 22% of all projects). They also have fewer
  external collaborators (0.4 vs. 1.9), slightly smaller teams and shorter
  descriptions, which fits recently started projects.

## Discussion

The scraper collected 129 MPIDR research projects from 14 research units on
26 October 2026. It loops over every unit's *Projects* tab and then opens each
project page. It then clicks through every unit's *Publications* tab, which
gives 1,710 distinct publications, against the 1,062 entries on project pages
(which also double-count papers linked to several projects). Publication
analyses should therefore use `mpidr_publications.csv`. The listed publications span 2004–2026, and 78% of projects with
publications have their first one in 2019 or later, so the data describe the
institute's current research.

All quality checks pass, and the missing values come from the website itself,
not from the scraper. Their severity differs:
- Missing **regions** (24%) is the main limitation. It is concentrated in the
  Social Demography lab, so comparisons of regional focus across labs should be
  read with caution.
- Missing **keywords** (6%) is small and mostly overlaps with missing regions.
- Missing **publication years** (11%) simply mark projects with no publications
  yet, most of them dissertations.

A typical project has about 3 MPIDR members and 1.7 external collaborators,
and lists about 8 publications, 82% of which are open access. The variance of
publications (about 83) is large relative to its mean, so a few projects
have far more publications than the rest. These patterns will be analysed in
Project 2.